# Part A – Legal & Courtesy Amount Detection
**ICS472 – Natural Language Processing**  
**Team:** Mohammed Al Sheqaih · Abdulrhman Ammar

**Goal:** Train a YOLOv8 object detector to locate two regions in each check image:
- **Class 0 – Legal amount** (handwritten Arabic text region)
- **Class 1 – Courtesy amount** (digit region)

**Evaluation metrics:** Accuracy @ IoU ≥ {0.50, 0.75, 0.90} and Mean IoU

**Output:** Bounding-box predictions on the test set + saved crops for Parts B and C

## 1. Imports & Setup

In [ ]:
import sys, os, shutil, json, random
sys.path.insert(0, os.path.abspath('.'))

import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch
from ultralytics import YOLO

from utils import (
    TRAIN_IMAGES, TEST_IMAGES, TRAIN_BBOX, TEST_BBOX, ARTIFACTS,
    parse_bbox, load_image, crop_region, yolo_to_xyxy, detection_metrics
)

# ── Device ──────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Device: {DEVICE}')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

plt.rcParams['figure.dpi'] = 120

YOLO_DIR  = ARTIFACTS / 'yolo_dataset'
CROPS_DIR = ARTIFACTS / 'crops'
RUNS_DIR  = ARTIFACTS / 'yolo_runs'

## 2. Prepare YOLO Dataset

YOLOv8 expects this layout:
```
yolo_dataset/
├── images/
│   ├── train/   (80 % of labelled images)
│   ├── val/     (20 % of labelled images)
│   └── test/
└── labels/
    ├── train/
    ├── val/
    └── test/
```
We use **symlinks** for images (avoids duplicating ~2 GB of TIFFs) and copy the small label `.txt` files.

`ac00048` is excluded — it has no bounding-box annotation.

In [ ]:
MISSING = {'ac00048'}  # no bbox label

def build_yolo_dirs():
    for split in ('train', 'val', 'test'):
        (YOLO_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
        (YOLO_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

def symlink_force(src, dst):
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    dst.symlink_to(src.resolve())

def populate_split(stems, split, img_src, lbl_src):
    for stem in stems:
        img_file = img_src / f'{stem}.tif'
        lbl_file = lbl_src / f'{stem}.txt'
        if img_file.exists():
            symlink_force(img_file, YOLO_DIR / 'images' / split / f'{stem}.tif')
        if lbl_file.exists():
            shutil.copy2(lbl_file, YOLO_DIR / 'labels' / split / f'{stem}.txt')

# ── Collect labelled train stems ──────────────────────────────────────────
all_train_stems = sorted(
    p.stem for p in TRAIN_BBOX.glob('*.txt')
    if p.stem not in MISSING
)
random.shuffle(all_train_stems)
split_idx     = int(0.8 * len(all_train_stems))
train_stems   = all_train_stems[:split_idx]
val_stems     = all_train_stems[split_idx:]
test_stems    = [p.stem for p in sorted(TEST_IMAGES.glob('*.tif'))]

print(f'Train: {len(train_stems)} | Val: {len(val_stems)} | Test: {len(test_stems)}')

# ── Build directory structure ─────────────────────────────────────────────
build_yolo_dirs()
populate_split(train_stems, 'train', TRAIN_IMAGES, TRAIN_BBOX)
populate_split(val_stems,   'val',   TRAIN_IMAGES, TRAIN_BBOX)
populate_split(test_stems,  'test',  TEST_IMAGES,  TEST_BBOX)

print('Dataset directories ready.')

In [ ]:
# ── Write data.yaml ───────────────────────────────────────────────────────
data_yaml = {
    'path':  str(YOLO_DIR.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    2,
    'names': {0: 'legal', 1: 'courtesy'},
}
yaml_path = YOLO_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print(f'data.yaml saved to: {yaml_path}')
print(open(yaml_path).read())

## 3. Train YOLOv8

We use **YOLOv8s** (small variant) pre-trained on COCO. Key choices:
- `imgsz=640` — good balance between speed and accuracy for this image size
- `epochs=100` — sufficient for fine-tuning; early stopping kicks in if val loss plateaus
- `batch=16` — adjust down to `8` if VRAM is tight
- `patience=20` — stops early if no improvement for 20 epochs

> **Runtime:** ~30–60 min on RTX 5060.

In [ ]:
model = YOLO('yolov8s.pt')  # downloads pretrained weights on first run

train_results = model.train(
    data    = str(yaml_path),
    epochs  = 100,
    imgsz   = 640,
    batch   = 16,          # reduce to 8 if OOM
    device  = DEVICE,
    patience= 20,
    seed    = SEED,
    project = str(RUNS_DIR),
    name    = 'yolov8s_checks',
    exist_ok= True,
    verbose = True,
)

BEST_WEIGHTS = RUNS_DIR / 'yolov8s_checks' / 'weights' / 'best.pt'
print(f'\nBest weights: {BEST_WEIGHTS}')

## 4. Validation Metrics (Built-in YOLO Report)

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
val_results = best_model.val(
    data   = str(yaml_path),
    split  = 'val',
    device = DEVICE,
    verbose= True,
)
print('mAP50  :', round(val_results.box.map50, 4))
print('mAP50-95:', round(val_results.box.map,  4))

## 5. Custom Evaluation on Test Set

We compute the project-specific metrics on the **test set**:
- Accuracy @ IoU ≥ 0.50 / 0.75 / 0.90
- Mean IoU

Both classes (legal + courtesy) are evaluated together as the assignment specifies.

In [ ]:
test_imgs = sorted(TEST_IMAGES.glob('*.tif'))

predictions   = []
ground_truths = []
pred_boxes_raw = []  # store raw xyxy predictions for crop saving

for img_path in tqdm(test_imgs, desc='Evaluating test set'):
    img     = Image.open(img_path)
    W, H    = img.size

    # ── Model prediction ──────────────────────────────────────────────
    result  = best_model(img_path, verbose=False)[0]
    pred    = {'courtesy': None, 'legal': None}
    pred_raw = {}
    for box in result.boxes:
        cls_id = int(box.cls)
        xyxy   = [round(v, 1) for v in box.xyxy[0].tolist()]
        key    = 'courtesy' if cls_id == 1 else 'legal'
        # keep the highest-confidence box per class
        if pred[key] is None:
            pred[key]     = xyxy
            pred_raw[key] = xyxy
    predictions.append(pred)
    pred_boxes_raw.append({'stem': img_path.stem, 'boxes': pred_raw, 'size': (W, H)})

    # ── Ground truth ──────────────────────────────────────────────────
    gt      = {'courtesy': None, 'legal': None}
    gt_path = TEST_BBOX / (img_path.stem + '.txt')
    if gt_path.exists():
        for cls, cx, cy, bw, bh in parse_bbox(gt_path):
            key       = 'courtesy' if cls == 1 else 'legal'
            gt[key]   = yolo_to_xyxy(cx, cy, bw, bh, W, H)
    ground_truths.append(gt)

# ── Compute metrics ───────────────────────────────────────────────────
metrics = detection_metrics(predictions, ground_truths, thresholds=(0.5, 0.75, 0.9))
print('\n── Test Set Detection Metrics ──')
for k, v in metrics.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Per-class IoU breakdown ────────────────────────────────────────────────
ca_ious, la_ious = [], []

from utils import iou
for pred, gt in zip(predictions, ground_truths):
    for key, lst in (('courtesy', ca_ious), ('legal', la_ious)):
        if pred[key] and gt[key]:
            lst.append(iou(pred[key], gt[key]))
        else:
            lst.append(0.0)

print(f'Courtesy — Mean IoU: {np.mean(ca_ious)*100:.2f}%')
print(f'Legal    — Mean IoU: {np.mean(la_ious)*100:.2f}%')

# ── Save metrics to disk ──────────────────────────────────────────────────
metrics_out = {
    **metrics,
    'courtesy_mean_iou': round(float(np.mean(ca_ious)) * 100, 2),
    'legal_mean_iou':    round(float(np.mean(la_ious)) * 100, 2),
}
with open(ARTIFACTS / 'partA_metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print('\nMetrics saved to artifacts/partA_metrics.json')

## 6. Visualise Predictions vs Ground Truth

In [ ]:
def draw_boxes(ax, img, pred, gt, title=''):
    ax.imshow(img, cmap='gray')
    styles = [
        ('pred',  pred, 'lime',  '-'),
        ('gt',    gt,   'red',   '--'),
    ]
    for label_prefix, boxes, color, ls in styles:
        for key, box in boxes.items():
            if box is None:
                continue
            x1, y1, x2, y2 = box
            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor=color, facecolor='none', linestyle=ls
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, f'{label_prefix}:{key}', color=color, fontsize=7)
    ax.axis('off')
    ax.set_title(title, fontsize=9)

fig, axes = plt.subplots(4, 1, figsize=(13, 16))
sample_idxs = [0, 5, 20, 50]
for ax, idx in zip(axes, sample_idxs):
    img = Image.open(test_imgs[idx])
    draw_boxes(ax, img, predictions[idx], ground_truths[idx],
               title=test_imgs[idx].stem)
plt.suptitle('Predictions (green) vs Ground Truth (red)', y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig(ARTIFACTS / 'partA_predictions.png', bbox_inches='tight')
plt.show()

## 7. Save Crops for Parts B and C

- **Train crops** — cropped using **ground-truth** bboxes (clean signal for training the recognisers)
- **Test crops**  — cropped using **model predictions** (simulates the real end-to-end pipeline)

Files are named `Cac#####.tif` (courtesy) and `Lac#####.tif` (legal) to match the annotation files.

In [ ]:
for split in ('train', 'test'):
    for region in ('courtesy', 'legal'):
        (CROPS_DIR / split / region).mkdir(parents=True, exist_ok=True)

def save_crop(img_path, cx, cy, bw, bh, out_path):
    img  = load_image(img_path)
    crop = crop_region(img, cx, cy, bw, bh)
    crop.save(out_path)

# ── Train crops (ground truth bboxes) ─────────────────────────────────────
train_img_paths = sorted(TRAIN_IMAGES.glob('*.tif'))
skipped = 0
for img_path in tqdm(train_img_paths, desc='Saving train crops'):
    if img_path.stem in MISSING:
        skipped += 1
        continue
    lbl_path = TRAIN_BBOX / (img_path.stem + '.txt')
    if not lbl_path.exists():
        skipped += 1
        continue
    for cls, cx, cy, bw, bh in parse_bbox(lbl_path):
        region  = 'courtesy' if cls == 1 else 'legal'
        prefix  = 'C' if cls == 1 else 'L'
        out     = CROPS_DIR / 'train' / region / f'{prefix}{img_path.stem}.tif'
        save_crop(img_path, cx, cy, bw, bh, out)
print(f'Train crops done. Skipped: {skipped}')

In [ ]:
# ── Test crops (model predictions) ────────────────────────────────────────
missing_preds = 0
for entry in tqdm(pred_boxes_raw, desc='Saving test crops'):
    stem     = entry['stem']
    img_path = TEST_IMAGES / f'{stem}.tif'
    img      = load_image(img_path)
    W, H     = img.size

    for key, (cls_id, prefix) in {
        'courtesy': (1, 'C'),
        'legal':    (0, 'L'),
    }.items():
        box = entry['boxes'].get(key)
        if box is None:
            # fall back to ground truth if model missed the box
            gt_path = TEST_BBOX / f'{stem}.txt'
            if gt_path.exists():
                for c, cx, cy, bw, bh in parse_bbox(gt_path):
                    if c == cls_id:
                        box = yolo_to_xyxy(cx, cy, bw, bh, W, H)
                        break
            if box is None:
                missing_preds += 1
                continue
        x1, y1, x2, y2 = [int(v) for v in box]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)
        crop    = img.crop((x1, y1, x2, y2))
        out     = CROPS_DIR / 'test' / key / f'{prefix}{stem}.tif'
        crop.save(out)

print(f'Test crops done. Missing predictions (fell back to GT): {missing_preds}')

## 8. Summary

In [ ]:
train_ca_crops = list((CROPS_DIR / 'train' / 'courtesy').glob('*.tif'))
train_la_crops = list((CROPS_DIR / 'train' / 'legal').glob('*.tif'))
test_ca_crops  = list((CROPS_DIR / 'test'  / 'courtesy').glob('*.tif'))
test_la_crops  = list((CROPS_DIR / 'test'  / 'legal').glob('*.tif'))

print('── Part A Complete ──')
print(f"  Acc @ IoU≥0.50 : {metrics['acc@0.5']}%")
print(f"  Acc @ IoU≥0.75 : {metrics['acc@0.75']}%")
print(f"  Acc @ IoU≥0.90 : {metrics['acc@0.9']}%")
print(f"  Mean IoU       : {metrics['mean_iou']}%")
print()
print(f'  Crops saved:')
print(f'    Train courtesy : {len(train_ca_crops)}')
print(f'    Train legal    : {len(train_la_crops)}')
print(f'    Test  courtesy : {len(test_ca_crops)}')
print(f'    Test  legal    : {len(test_la_crops)}')
print()
print('  → Run 02_Part_B_Courtesy.ipynb next')